# Parte 1 — Engenharia de Dados: Camada Bronze

Carrega os cinco CSVs e a cotação do dólar em tabelas Delta. Cada carga recebe ingestion_datetime e usa Append. A resposta do Banco Central utilizada na carga está armazenada em JSON no Volume.


## Configuração da camada Bronze

Usa o mesmo catálogo, schemas e caminho de entrada definidos na preparação do Lakehouse.


In [0]:
# Mantém os caminhos iguais aos do notebook de preparação.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

## Carga dos CSVs — Tabelas Bronze

Lê os CSVs com cabeçalho e mantém os campos como texto. Grava tb_movies_info, tb_movies_financials, tb_movies_metrics, tb_credits_and_tags e tb_movies_reviews em Delta, com o timestamp da carga. Append acrescenta registros a cada execução.


In [0]:
# A limpeza e a conversão dos dados serão realizadas na camada Silver.

from pyspark.sql.functions import current_timestamp

# 1. Mapeamento dos caminhos dos arquivos no Volume
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Opções de leitura: manter as colunas como texto na Bronze
df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=False)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=False)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=False)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=False)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=False)

# 3. Gravação em Delta com timestamp de ingestão
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")

## API do Banco Central — Parâmetros do período

Cria os parâmetros de data inicial e final para consultar os últimos sete dias corridos. As datas são enviadas à API no formato MM-DD-AAAA.


In [0]:
# Os parâmetros abaixo são usados pela tentativa de consulta à API.

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# Últimos 7 dias corridos, incluindo hoje
hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()

dbutils.widgets.text(
    "data_inicio",
    (hoje - timedelta(days=6)).strftime("%m-%d-%Y"),
    "Data inicial (MM-DD-AAAA)"
)

dbutils.widgets.text(
    "data_fim",
    hoje.strftime("%m-%d-%Y"),
    "Data final (MM-DD-AAAA)"
)

## API do Banco Central — Tentativa de conexão

Consulta a API PTAX com o período escolhido e solicita dataHoraCotacao e cotacaoCompra. A tentativa de conexão ficou registrada; a carga usada neste projeto vem do arquivo JSON da próxima etapa.


In [0]:
# Esta célula registra a tentativa de acesso direto à API do Banco Central.

import requests
from pyspark.sql.functions import current_timestamp

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# Consultar a API
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

resposta = requests.get(url, params={
    "@dataInicial": f"'{data_inicio}'",
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}, timeout=30)

# Tentativa falha!

---------------------------------------------------------------------------
gaierror                                  Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/urllib3/connection.py:198, in HTTPConnection._new_conn(self)
    197 try:
--> 198     sock = connection.create_connection(
    199         (self._dns_host, self.port),
    200         self.timeout,
    201         source_address=self.source_address,
    202         socket_options=self.socket_options,
    203     )
    204 except socket.gaierror as e:

File /databricks/python/lib/python3.12/site-packages/urllib3/util/connection.py:60, in create_connection(address, timeout, source_address, socket_options)
     58     raise LocationParseError(f"'{host}', label empty or too long") from None
---> 60 for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
     61     af, socktype, proto, canonname, sa = res

File /usr/lib/python3.12/socket.py:963, in getaddrinfo(host, port, f

## Carga da cotação — bronze.tb_cotacao_dolar

Lê cotacao_dolar.json no Volume, desmembra a lista value e grava as cotações em Delta Append, adicionando ingestion_datetime. O arquivo contém a resposta obtida da API; os parâmetros da consulta não atualizam esse arquivo.


In [0]:
# Carrega a resposta da API que foi disponibilizada no Volume.

# Leitura da resposta da API armazenada no Volume
df_cotacao_dolar_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{landing_path}/cotacao_dolar.json")
    .selectExpr("explode(value) AS cotacao")
    .select("cotacao.dataHoraCotacao", "cotacao.cotacaoCompra")
)

# Gravação na Bronze
df_cotacao_dolar_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_cotacao_dolar")

display(df_cotacao_dolar_raw)

dataHoraCotacao,cotacaoCompra
2026-10-01 13:10:35.4469,5.2073
2026-10-02 13:03:16.256632,5.2232
2026-10-05 13:07:36.558602,4.9853
2026-10-06 13:03:21.267287,4.9692
2026-10-07 13:05:23.249063,4.9929
